# OpenShell sandboxed deployment demo

Walks the create, package, deploy, and invoke flow for an agent deployed as a policy-governed OpenShell sandbox on a local NeMo Platform.

How to run this:

- Use a python3 kernel from the repo's workspace env, the one with the `nemo_platform` SDK installed (e.g. the repo `.venv`).
- The platform-API steps use the NeMo Python SDK (`NeMoPlatform`), the agent invoke uses `httpx`, and the egress check shells out to the `openshell` CLI.
- Two services must be running first, each in its own terminal (the OpenShell gateway and `nemo services run`). See "Before you begin". The notebook runs only the steps that terminate.

## What it shows

An ordinary NAT ReAct agent is deployed through the standard NeMo deployments API, but the executor is `openshell-local`, so the agent runs inside an OpenShell sandbox under a generated SandboxPolicy:

- Landlock filesystem/process isolation. The agent runs as the unprivileged `sandbox` user with a default-deny filesystem allowlist.
- Zero-egress model access. The agent reaches models through OpenShell's gateway-managed `inference.local` route, so the sandbox is granted no direct network egress. The model call is brokered over the supervisor's gateway channel rather than through a policy egress rule; a call from the sandbox to anywhere else (e.g. `example.com`) is blocked at the boundary.

The same agent image, deployed unchanged, gets its model access brokered by the gateway, and the sandbox itself can reach nothing on the network directly.

## Before you begin

Everything you need on this machine:

1. Docker, since the OpenShell gateway and the sandboxes run as containers.
2. The `openshell` CLI on your PATH (from the OpenShell install).
3. The repo workspace env with the deployments backend's `openshell` extra. A bootstrapped workspace already has it, and the cell below installs it. Packaging (Step 1) also needs the agents `container` extra; if `nemo agents package` reports a missing `python-on-whales`, run `uv pip install -e 'plugins/nemo-agents[container]'`.
4. Two services running, each in its own terminal (both are long-running foreground processes): the OpenShell gateway on `:17670` (Prerequisite 1) and the NeMo platform with the `openshell-local` executor on `:8080` (Prerequisite 2).
5. A model your Inference Gateway serves. `nemo models list` shows the options; you pick one and wire `inference.local` before Step 1.

The `openshell` SDK (`openshell>=0.0.83`) is a normal PyPI dependency of the plugin's `[openshell]` extra, and a workspace `uv sync` resolves it. Then run the cells top to bottom.

In [ ]:
%%bash
# install the deployments backend's openshell extra (a bootstrapped workspace already has it)
uv sync --package nemo-deployments-plugin --extra openshell

## Prerequisites

The two services from **Before you begin**, in detail. See `SETUP.md` at the repo root for more on the platform; the gateway is fully set up below.

### 1. OpenShell gateway on `:17670` (separate terminal)

`:17670` is OpenShell's documented docker default. It must NOT be `:8080`: that belongs to the NeMo platform. Two things steer you onto `:8080` and cause the collision, so avoid both: OpenShell's gateway container image defaults its CMD to `--port 8080` (the `deploy/docker` compose file publishes `:8080` unchanged), and the CLI's throwaway `openshell gateway add http://127.0.0.1:8080` example.

Use the compose file shipped in this directory (`docker-compose.yml` + `gateway.toml`), pinned to the docker driver, plaintext, on `:17670`. Run these **in their own terminal**:

```bash
# one-time: generate the JWT signing keys the docker driver needs to mint
# sandbox tokens (writes /var/lib/openshell/tls/jwt/*). Re-run only if the
# /var/lib/openshell state dir is wiped.
docker run --rm --user 0 -v /var/lib/openshell:/var/lib/openshell \
  ghcr.io/nvidia/openshell/gateway:0.0.92 generate-certs \
  --output-dir /var/lib/openshell/tls \
  --server-san 127.0.0.1 --server-san localhost --server-san host.openshell.internal

docker compose -f plugins/nemo-deployments/examples/openshell/docker-compose.yml up -d
openshell gateway add http://127.0.0.1:17670 --local --name docker-dev
```

Tear the gateway down after the demo with `docker compose -f plugins/nemo-deployments/examples/openshell/docker-compose.yml down`.

The cell below **verifies** the CLI reaches the gateway. The endpoint column must read `:17670`, not `:8080`.

In [ ]:
%%bash
openshell gateway list          # * docker-dev  http://127.0.0.1:17670  ... plaintext
openshell sandbox list          # reaches the gateway (e.g. "No sandboxes found.")

### 2. NeMo platform with the `openshell-local` executor (separate terminal)

The executor lives in `packages/nmp_platform/config/local.yaml`, which is NOT the config `nemo services run` loads by default (that bundled default has no `deployments:` block at all). You MUST pass it with `--config`, or the executor never registers and every deploy below fails with an unknown-executor error. The registered block:

```yaml
deployments:
  executors:
    - name: openshell-local
      backend: openshell
      config:
        gateway_endpoint: http://127.0.0.1:17670
        serve_workdir: /home/sandbox     # sandbox-writable CWD for `nat serve`
        platform_egress: null            # agent uses inference.local (gateway-managed), so the
                                         # sandbox needs NO direct egress -> pure default-deny
  # No default_executor: sandbox is opt-in; the deploy in Step 3 names it.
```

Start the platform **in its own terminal**, bound to `0.0.0.0`. That matters: the `inference.local` route is dialed from the sandbox, not from the gateway process. The docker driver writes a literal `172.18.0.1 host.openshell.internal` into each sandbox's `/etc/hosts` (the sandbox network's gateway address), so the platform has to be listening there; on `--host 127.0.0.1` every model call fails with a 503. It also puts an unauthenticated dev platform on every interface, LAN included, so do not do this on an untrusted network. This process runs in the foreground and does not return, which is why it does not live in a notebook cell:

```bash
export NMP_BASE_URL=http://localhost:8080
nemo services run --host 0.0.0.0 --port 8080 \
  --config packages/nmp_platform/config/local.yaml
```

### 3. A sandbox-compatible agent image

Built in Step 1 below. Packaging needs the agents plugin `container` extra (`python-on-whales`). If `nemo agents package` fails with a missing `python-on-whales`, install the extra (this is the targeted fix; `uv sync --all-extras` also works):

```bash
uv pip install -e 'plugins/nemo-agents[container]'
```

### Build the SDK client

Set `NMP_BASE_URL` once (the `nemo` CLI cells below read it too), then build one `NeMoPlatform` client in **direct mode**: passing `base_url` explicitly skips config-file bootstrap and auth injection, which the local platform needs none of. Every platform-API cell below reuses this `client`.

In [ ]:
%env NMP_BASE_URL=http://localhost:8080

In [ ]:
import json
import os
import subprocess

import httpx
from nemo_platform import NeMoPlatform

WORKSPACE = "default"
# Direct mode: an explicit base_url means no config bootstrap and no auth headers
# (fine for the local platform). Honors the NMP_BASE_URL set above.
client = NeMoPlatform(base_url=os.environ["NMP_BASE_URL"], workspace=WORKSPACE)

# The deployments API has no typed SDK resource yet, so we drive it through the
# client's untyped raw-request escape hatch against this base path.
DEPLOY = f"/apis/deployments/v2/workspaces/{WORKSPACE}"
print("platform:", client.base_url)

In [ ]:
# readiness gate for the platform started in prerequisite 2
r = httpx.get(f"{os.environ['NMP_BASE_URL']}/health/ready", timeout=5)
r.raise_for_status()
print(r.json())  # {'status': 'ready'}

## Pick a model, then wire inference.local

The agent reaches models through OpenShell's gateway-managed **`inference.local`** route: the gateway routes `https://inference.local/v1` to a registered provider (the platform Inference Gateway) and injects the real credential, so the sandbox needs **no direct egress**. First pick a model the Inference Gateway serves, then do the one-time operator wiring on the gateway.

In [ ]:
%%bash
# pick a model the Inference Gateway serves; you set it as MODEL below
nemo models list

In [ ]:
MODEL = "default/openai-openai-gpt-4o-mini"  # <- set to a model `nemo models list` shows above
os.environ["MODEL"] = MODEL  # so the operator-wiring cell below can read it
print("model:", MODEL)

**Operator setup (once per gateway).** Register the platform Inference Gateway as an `openai` provider and bind `inference.local` to it. `host.openshell.internal` is the gateway's alias for its own host (where the platform listens). The credential is a dummy, since the gateway injects the real one. `inference set` verifies the endpoint gateway-side.

In [ ]:
%%bash
openshell provider create --name nemo-igw --type openai \
  --credential OPENAI_API_KEY=empty \
  --config OPENAI_BASE_URL=http://host.openshell.internal:8080/apis/inference-gateway/v2/workspaces/default/openai/-/v1 \
  || echo "(provider may already exist; use 'openshell provider update' to change its config)"
openshell inference set --provider nemo-igw --model "$MODEL"
openshell inference get

## Step 1: package the agent as a sandbox image

The agent config is a checked-in, greppable file, `agent/config.yaml`, next to this notebook. It is a normal NAT ReAct config, already sandbox-ready (LLM pointed at `https://inference.local/v1`, `nemo_files` telemetry tracer removed), with the model id left as a deploy-time param.

`nemo agents package` uses that file's folder as the Docker build context. The `agent/` folder holds only `config.yaml`, so the context stays clean, and it bakes to `/workspace/config.yaml`, the path Step 2's serve command loads. The model is injected at deploy via `nat serve --override llms.llm.model_name <MODEL>`, and nothing is written to `/tmp`.

Build the image with the OpenShell runtime profile (adds the `sandbox` user, etc.).

The `--sandbox-runtime openshell` flag renders a provider-neutral sandbox profile into the image build. The config is baked to `/workspace/config.yaml`, which is exactly what Step 2's serve command loads.

In [ ]:
%%bash
nemo agents package \
  --agent  agent/config.yaml \
  --nat-version 1.8.0 \
  --sandbox-runtime openshell \
  --tag    nemo-agent-igw:test

## Step 2: create the DeploymentConfig

Describe the container: the image, the `nat serve` command, and the port. The serve command is what the backend runs inside the sandbox. The model id is injected via `--override`, and the agent reaches the LLM through `inference.local`, so there is no proxy handling or egress plumbing here.

`client.post(..., cast_to=object)` returns the parsed JSON as a dict. A non-2xx response (e.g. a duplicate name) raises `APIStatusError` whose message is the response body, so the error is loud, not swallowed.

In [ ]:
cfg = client.post(
    f"{DEPLOY}/deployment-configs",
    cast_to=object,  # untyped raw request -> parsed JSON dict
    body={
        "name": "igw-agent-cfg",
        "containers": [
            {
                "name": "agent",
                "image": "nemo-agent-igw:test",
                "command": [
                    "/workspace/.venv/bin/nat",
                    "serve",
                    "--config_file",
                    "/workspace/config.yaml",
                    "--override",
                    "llms.llm.model_name",
                    MODEL,  # deploy-time model id (see above)
                    "--host",
                    "0.0.0.0",
                    "--port",
                    "9000",
                ],
                "ports": [{"containerPort": 9000, "name": "http"}],
            }
        ],
    },
)
print(json.dumps(cfg, indent=2))

## Step 3: deploy (executor = openshell-local)

Create the Deployment. Sandbox is opt-in, so you must name the executor. There is no default sandbox executor, which is deliberate: ordinary deployments and the e2e harness keep their normal docker/k8s path.

Behind the scenes the `OpenShellDeploymentBackend`:

1. Generates a SandboxPolicy: default-deny filesystem (`read_only: /opt` for the interpreter; `read_write: /home/sandbox, /tmp, /dev/shm` for the Dask runtime `nat serve` spins up), `run_as_user: sandbox`, and a single egress rule to `platform_egress` (the Inference Gateway). A mandatory `nemo_platform` egress rule is always injected, so a hand-written policy can never sever the sandbox's path back to the platform.
2. Creates the sandbox from the image and applies the policy.
3. Runs the serve command in the sandbox-writable `serve_workdir` (`/home/sandbox`). The image legitimately owns `/workspace` as its `agent` user, which the `sandbox` user cannot write.
4. Exposes the service through the gateway and reports the URL.

In [ ]:
dep = client.post(
    f"{DEPLOY}/deployments",
    cast_to=object,
    body={
        "name": "igw-agent",
        "deployment_config": "igw-agent-cfg",
        "executor": "openshell-local",
    },
)
print(json.dumps(dep, indent=2))

## Step 4: wait for READY and read the endpoint

The reconciler drives the deployment from `PENDING` to `STARTING` to `READY` (typically well under a minute). Re-run the poll cell until it prints `READY`.

In [ ]:
# poll status -- re-run until this prints READY
dep = client.get(f"{DEPLOY}/deployments/igw-agent", cast_to=object)
print(dep["status"])

In [ ]:
# grab the exposed endpoint once READY; `url` persists to later cells
dep = client.get(f"{DEPLOY}/deployments/igw-agent", cast_to=object)
url = dep["endpoints"][0]["url"]
print(url)  # e.g. http://<sandbox>--<svc>.openshell.localhost:17670/

## Step 5: invoke the agent

Ask a question that forces both a tool call and an LLM call. The agent's LLM call goes to `inference.local`, which the gateway routes to the platform Inference Gateway, so the sandbox makes no direct egress. This cell hits the agent's own NAT-served endpoint (via the openshell gateway), not the platform API, so it is a plain `httpx` POST rather than the SDK client.

Other routes the served workflow exposes: `POST /v1/chat/completions`, `/chat`, `/v1/workflow`, and `/generate/stream`.

In [ ]:
dep = client.get(f"{DEPLOY}/deployments/igw-agent", cast_to=object)
url = dep["endpoints"][0]["url"]
r = httpx.post(
    f"{url}generate",
    json={"input_message": "What is the current date and time? Use your current_datetime tool"},
    timeout=60,
)
r.raise_for_status()
print(r.json())  # {'value': 'The current date and time is 2026-... +0000.'}

### Debugging a 502 on invoke

A 502 means the gateway reached the sandbox but the agent's `nat serve` (port 9000) isn't answering, because it crashed or hasn't come up. `READY` does not gate on serve health (the detached serve exec is unsupervised), so:

1. Give it a moment and retry the invoke. `nat serve` plus its Dask runtime can take 30-60s after `READY`.
2. If it persists, read the serve log inside the sandbox. The backend launches `nat serve` with its output redirected to `/tmp/nemo-serve.log`, and the cell below cats it.

The most common root cause is that the `MODEL` set in Step 1 isn't served on your Inference Gateway (`nemo models list`), so `nat serve` fails at startup. Next-most-common is that the config didn't bake to `/workspace/config.yaml`.

In [ ]:
# read the agent's serve log from inside the sandbox to see why it 502s
dep = client.get(f"{DEPLOY}/deployments/igw-agent", cast_to=object)
sbx = dep["endpoints"][0]["url"].split("://", 1)[-1].split("--", 1)[0]
res = subprocess.run(
    ["openshell", "sandbox", "exec", "--name", sbx, "--", "cat", "/tmp/nemo-serve.log"],
    capture_output=True,
    text=True,
    stdin=subprocess.DEVNULL,
    timeout=30,
)
print(res.stdout or "(serve log empty -- process may not have started yet)")
print(res.stderr, end="")

### (Optional) prove the egress policy

Show that the sandbox can reach nothing on the network directly. The `example.com` call is blocked by the policy default-deny. Step 5's LLM call still worked, because it went through the gateway-managed `inference.local` route rather than sandbox egress. A nonzero exit from the sandboxed `curl` is the PASS.

In [ ]:
# sandbox name is nmp-<hash>; derive it from the endpoint URL (or read `openshell sandbox list`)
dep = client.get(f"{DEPLOY}/deployments/igw-agent", cast_to=object)
sbx = dep["endpoints"][0]["url"].split("://", 1)[-1].split("--", 1)[0]
print("sandbox:", sbx)

# A NONZERO exit from the inner curl is the PASS: example.com must be blocked by the
# default-deny egress policy. The inner `curl -m 5` self-aborts, and the outer timeout
# is a backstop so a wedged `exec` can never hang the notebook.
try:
    res = subprocess.run(
        ["openshell", "sandbox", "exec", "--name", sbx, "--", "curl", "-sS", "-m", "5", "https://example.com"],
        capture_output=True,
        text=True,
        stdin=subprocess.DEVNULL,
        timeout=30,
    )
    print(
        "BLOCKED as expected (policy default-deny)"
        if res.returncode != 0
        else f"UNEXPECTED: egress to example.com succeeded\n{res.stdout}"
    )
except subprocess.TimeoutExpired:
    print("exec timed out (30s) -- inconclusive; check `openshell sandbox list`")
# meanwhile Step 5's LLM call went via inference.local (gateway-routed; no sandbox egress)

## Step 6: clean up

In [ ]:
client.delete(f"{DEPLOY}/deployments/igw-agent", cast_to=httpx.Response)  # 204 -> tears down the sandbox
client.delete(f"{DEPLOY}/deployment-configs/igw-agent-cfg", cast_to=httpx.Response)
print("deleted deployment + config")

## Known limitations

- The NAT config is baked into the image; deploy-time config injection is a follow-up.
- `inference.local` requires the one-time operator wiring (provider + `inference set`) per gateway; the deployment backend does not auto-register it yet.
- The deployments API has no typed SDK resource yet, so the platform-API cells use the untyped `client.post(..., cast_to=object)`; when the typed resource lands these become `client.deployments.*`.

## Don't forget the gateway

After the demo, tear the gateway down in its terminal:

```bash
docker compose -f plugins/nemo-deployments/examples/openshell/docker-compose.yml down
```